In [ ]:
#| default_exp game/food

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.plot.cube import HexPosition, field_of_view

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import math
from dataclasses import dataclass, field
from functools import cached_property
#| export
import heapq

In [ ]:
#| export

from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from scipy.optimize import linear_sum_assignment
from HexMagic.water.soil import SoilSystem
from enum import Enum

In [ ]:
myTerr = TerraDemo().japan_korea_map()
rivers = myTerr.carve_to_ocean(num_lakes=0)

In [ ]:
myTerr.compute_climate()

In [ ]:
basins = DrainageBasins(myTerr)

In [ ]:
#TerrainDisplay(TerrainOverlay(),ClimateOverlay(),terrain=myTerr)

flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

# Three tiers with their typical scales
tiers = [
    ('board',  0.5,  'solid fill'),
    ('list',   1.5,  'simple pattern'),
    ('large',  3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx = 80 + i * spacing
    cy = 90
    pid = f"queen_{size}_{i}"

    # Draw backdrop circle scaled to piece size
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    # Draw the queen using draw_piece
    flag.draw_piece(
        PieceType.ROOK, MapCord(cx, cy), canvas,
        scale=scale, size=size, piece_id=pid, layer=f"queen_{i}",
    )

    # Labels
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Queen at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()

## FoodYield

In [ ]:
#| export


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve: dict = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
# Make sure soil exists
from HexMagic.water.soil import SoilSystem
soil = SoilSystem.from_plates(myTerr, [])

fy = FoodYield(myTerr, basins)
fy.compute()
fy.summary()


I think this is ok. there is a bunch of mountains and oceans

In [ ]:
fy = FoodYield(myTerr, basins)
fy.compute()
fy.summary()

Thoughts

lets build the overlay. I know we need terrainPatterns for the dots. What would be a good color to use. I am assuming we will use it either with a cream backgroun or the elevations. does it make sense to have different colors or just different densities

In [ ]:
#| export


def food_overlay(fy: FoodYield, color: str = "#558B2F") -> str:
    """Build dotted overlay from computed food yield tiers."""
    terrain = fy.terrain
    grid = terrain.hexGrid
    
    patGen = TerrainPatterns(terrain)
    patterns = patGen.ballDensity(
        levels=fy.n_tiers,
        fills=[color],
        prefix="food_yield"
    )
    
    overlay = ""
    used = set()
    
    for i in range(len(terrain.elevations)):
        if terrain.elevations[i] <= 0:
            continue
        tier = int(fy.tiers[i])
        if tier == 0:
            continue  # Skip barren — let base map show through
        
        used.add(tier)
        pat_name = patterns[tier].attributes['id']
        hex_obj = grid.hexes[i]
        
        pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
        overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
    
    for tier in sorted(used):
        grid.builder.add_definition(patterns[tier])
    
    return overlay


In [ ]:
svg = food_overlay(fy)
myTerr.hexGrid.builder.layers = []
myTerr.terrainCream()
myTerr.hexGrid.builder.adjust("food", svg)
show(myTerr.hexGrid.builder)


## FoodOverlay

In [ ]:
#| export
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)


In [ ]:
!cat ../../HexMagic/game/piece.py

So we now have the basics to have food on the map we should work out the mechanics of how pieces move to food harvest and share food with one another. 

1. we need a dystyra path to pick a target
2. something that will assign a piece
3. the share mechanism. my idea for this is the same a the context where they have a cone coming out from them a sight modified number of hexes. feeding priority is given by rank and then who has the least amount of food. you will always keep 4 turns of your diet of food so you won't starve yourself.

Other things we should add?

In [ ]:
??Piece

So we have food, diet and food_capacity on pieces. If starving you start to loose health. all health lost you die

Diet-proportional

Lets do 5, I am a generous king.

harvest_strength × tile_tier

In [ ]:
#!cat ../../HexMagic/plot/*.py

So we have quite a bit of math that uses hexpostions to find adjacent hexes

allowed_rotations=1

 KING > QUEEN > ROOK > BISHOP > KNIGHT > PAWN

In theory each piece does its thing in isolation. We are going to build rules a level up, but at the piece level if told to give you give.

How much of this can you build?

lets extract out a pieces in sight with a multipler (pieces at higher elevations are going to see farther, but we can figure out by home much by later). it gives us a nice board balance where the higher ground has strategic advantage, but the lower ground has more food.

We now have
```
def field_of_view(origin: HexPosition, facing: HexPosition, max_distance: int) -> list[HexPosition]:
    """Get all hexes within a 120° field of view (±60° from facing) up to max_distance."""
    results = []
    for radius in range(1, max_distance + 1):
        for hex_pos in origin.ring(radius):
            if hex_in_cone(hex_pos - origin, facing, 1):
                results.append(hex_pos)
    return results
```

In [ ]:
#| export
from HexMagic.plot.cube import HexPosition, field_of_view

STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

# Lower index = higher priority (fed first)
RANK_ORDER = {
    PieceType.KING:   0,
    PieceType.QUEEN:  1,
    PieceType.ROOK:   2,
    PieceType.BISHOP: 3,
    PieceType.KNIGHT: 4,
    PieceType.PAWN:   5,
}


def harvest(piece: Piece, grid: HexGrid, food_tiers: np.ndarray):
    """HARVEST action: gather food from current tile."""
    if piece.location is None or piece.location < 0:
        return
    tier = int(food_tiers[piece.location])
    gained = piece.harvest_strength * tier
    piece.food = min(piece.food + gained, piece.food_capacity)



def eat(piece: Piece) -> bool:
    """Consume food; apply starvation damage. Returns True if alive."""
    shortfall = max(0.0, piece.diet - piece.food)
    piece.food = max(0.0, piece.food - piece.diet)
    if shortfall > 0:
        piece.health -= shortfall * STARVE_MULT
    return piece.health > 0


def piece_food_step(piece: Piece, instruction: 'Instruction',
                    grid: HexGrid, food_tiers: np.ndarray,
                    all_pieces: list[Piece]) -> bool:
    """Run one turn's food cycle for a single piece.
    
    Call AFTER movement has been resolved.
    Returns True if piece is still alive.
    """
    if instruction == Instruction.HARVEST:
        harvest(piece, grid, food_tiers)
    elif instruction == Instruction.GIVE:
        give(piece, grid, all_pieces)

    return eat(piece)


In [ ]:
@patch
def pieces_in_sight(piece: Piece, grid: HexGrid, elevations: np.ndarray,
                    squads: list[Squad],
                    elevation_mult: float = 0.005,
                    facing_only: bool = False,
                    allied_only: bool = False) -> list[Squad]:
    """Find pieces visible from piece's location, grouped by squad.
    
    Accepts squads, returns squads containing only visible members.
    Empty squads (no visible members) are dropped.
    """
    if piece.location is None or piece.location < 0:
        return []

    elev = max(0, elevations[piece.location])
    effective_sight = int(piece.sight + elev * elevation_mult)
    origin = grid.index_to_hexposition(piece.location)
    facing_dir = HexPosition.directions()[piece.facing % 6]

    if facing_only:
        fov_set = set(field_of_view(HexPosition.origin(), facing_dir, effective_sight))

    result = []
    for squad in squads:
        visible_members = []
        for other in squad.alive:
            if other is piece or other.location is None:
                continue
            if allied_only and other.owner_id != piece.owner_id:
                continue

            relative = grid.index_to_hexposition(other.location, piece.location)

            if facing_only:
                if relative not in fov_set:
                    continue
            else:
                if abs(relative) > effective_sight:
                    continue

            visible_members.append(other)

        if visible_members:
            result.append(Squad(id=squad.id, name=squad.name, pieces=visible_members))

    return result


In [ ]:
??Squad

In [ ]:
directions_in_cone??

can pieces_in_ sight take and give back squads

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure — the natural shape is: flatten squads in, check visibility on each piece, then regroup the visible ones back into their source squads (dropping any that end up empty). Here's a clean version:

```python
@patch
def pieces_in_sight(piece: Piece, grid: HexGrid, elevations: np.ndarray,
                    squads: list[Squad],
                    elevation_mult: float = 0.005,
                    facing_only: bool = False,
                    allied_only: bool = False) -> list[Squad]:
    """Find pieces visible from piece's location, grouped by squad.
    
    Accepts squads, returns squads containing only visible members.
    Empty squads (no visible members) are dropped.
    """
    if piece.location is None or piece.location < 0:
        return []

    elev = max(0, elevations[piece.location])
    effective_sight = int(piece.sight + elev * elevation_mult)
    origin = grid.index_to_hexposition(piece.location)
    facing_dir = HexPosition.directions()[piece.facing % 6]

    if facing_only:
        fov_set = set(field_of_view(HexPosition.origin(), facing_dir, effective_sight))

    result = []
    for squad in squads:
        visible_members = []
        for other in squad.alive:
            if other is piece or other.location is None:
                continue
            if allied_only and other.owner_id != piece.owner_id:
                continue

            relative = grid.index_to_hexposition(other.location, piece.location)

            if facing_only:
                if relative not in fov_set:
                    continue
            else:
                if abs(relative) > effective_sight:
                    continue

            visible_members.append(other)

        if visible_members:
            result.append(Squad(id=squad.id, name=squad.name, pieces=visible_members))

    return result
```

Key points:

- **Input**: `list[Squad]` — works with the squad structure you already have
- **Output**: `list[Squad]` — each returned squad is a *filtered copy* containing only visible members. The `id` and `name` are preserved so the caller knows which squad they came from
- **Empty squads dropped** — if nobody in a squad is visible, it's not in the result
- A single piece not in a squad can just be wrapped: `[Squad(pieces=[lone_piece])]`

If you also want a convenience flat version (e.g. for `give` which just needs a flat piece list), you could add a one-liner:

```python
def visible_pieces_flat(piece, grid, elevations, squads, **kw) -> list[Piece]:
    """Flat list of all visible pieces across squads."""
    return [p for sq in piece.pieces_in_sight(grid, elevations, squads, **kw)
              for p in sq.pieces]
```

In [ ]:
@patch
def give(piece: Piece, grid: HexGrid, elevations: np.ndarray,
         battlefield: list[Piece]):
    """GIVE action: distribute surplus food to allies in facing cone.
    
    battlefield is ALL pieces (across all squads) — give targets
    any allied piece in sight, not just squad-mates.
    """
    reserve = FOOD_RESERVE_TURNS * piece.diet
    surplus = max(0.0, piece.food - reserve)
    if surplus <= 0:
        return

    receivers = [
        p for p in piece.pieces_in_sight(
            grid, elevations, battlefield,
            facing_only=True, allied_only=True
        )
        if p.food < p.food_capacity
    ]

    receivers.sort(key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

    for receiver in receivers:
        if surplus <= 0:
            break
        amount = min(surplus, receiver.food_capacity - receiver.food)
        receiver.food += amount
        piece.food -= amount
        surplus -= amount


## TroopPath

In [ ]:
@dataclass
class TroopPath:
    """A resolved hex route with cached metadata."""
    hexes: list[int]
    cost: float = 0.0          # total movement cost (sum of _move_cost steps)
    
    def __len__(self):   return len(self.hexes)
    def __iter__(self):  return iter(self.hexes)
    def __bool__(self):  return len(self.hexes) > 1   # empty or single-hex = falsy
    def __getitem__(self, i): return self.hexes[i]
    
    @cached_property
    def as_set(self) -> set[int]:
        return set(self.hexes)
    
    @property
    def start(self) -> int: return self.hexes[0]
    
    @property
    def end(self) -> int:   return self.hexes[-1]
    
    def to_rules(self, grid: HexGrid, start_facing: int = 0) -> list[int]:
        return InstructionList.path_to_rules(self.hexes, grid, start_facing=start_facing)


In [ ]:
#| export
@patch
def pathfind(self: Piece, target: int, grid: HexGrid,
             elevations: np.ndarray,
             countries: np.ndarray = None) -> TroopPath:
    """Dijkstra from piece.location to target. Returns TroopPath (falsy if unreachable)."""
    start = self.location
    if start is None or start < 0:
        return TroopPath([], 0.0)

    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    best = {start: 0.0}

    while pq:
        cost, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        if current == target:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return TroopPath(path, cost)

        for nb in grid.neighborsOf(current):
            if nb in visited or nb in grid.invalidRegion:
                continue
            if elevations[nb] <= 0:
                continue
            if countries is not None and countries[nb] < 0:
                continue
            new_cost = cost + _move_cost(elevations, current, nb)
            if new_cost < best.get(nb, float('inf')):
                best[nb] = new_cost
                parent[nb] = current
                heapq.heappush(pq, (new_cost, nb))

    return TroopPath([], 0.0)





In [ ]:
#| export
@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> TroopPath:
    """Nearest reachable tile with food >= min_tier. Returns TroopPath (falsy if none)."""
    start = self.location
    if start is None or start < 0:
        return TroopPath([], 0.0)

    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    best = {start: 0.0}

    while pq:
        cost, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        if current != start and int(food_tiers[current]) >= min_tier:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return TroopPath(path, cost)

        for nb in grid.neighborsOf(current):
            if nb in visited or nb in grid.invalidRegion:
                continue
            if elevations[nb] <= 0:
                continue
            if countries is not None and countries[nb] < 0:
                continue
            new_cost = cost + _move_cost(elevations, current, nb)
            if new_cost < best.get(nb, float('inf')):
                best[nb] = new_cost
                parent[nb] = current
                heapq.heappush(pq, (new_cost, nb))

    return TroopPath([], 0.0)



In [ ]:
#| export

@patch
def pathfind_to_rules(self: Piece, target: int, grid: HexGrid,
                      elevations: np.ndarray,
                      countries: np.ndarray = None) -> InstructionList:
    """Thin wrapper: path to target → InstructionList."""
    path = self.pathfind(target, grid, elevations, countries)
    if not path:
        return InstructionList()
    return InstructionList(path.to_rules(grid, self.facing), cursor=0, patrol=False)

I added it. The feed turn is more complex since you will want to feed things outside of your squad (so you want a sight that goes outside your squad). but lets keep refactoring

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good point — `give` inherently needs cross-squad visibility. So the natural split is:

- **Squad** owns its own pieces and does intra-squad things (rank ordering, iteration, alive filtering)
- **Feed cycle** lives one level up where it can see all pieces across all squads

Here's what I'd refactor next. First, update `give` and `feed_turn` signatures to take a flat list of *all* battlefield pieces:

```python
@patch
def give(piece: Piece, grid: HexGrid, elevations: np.ndarray,
         battlefield: list[Piece]):
    """GIVE action: distribute surplus food to allies in facing cone.
    
    battlefield is ALL pieces (across all squads) — give targets
    any allied piece in sight, not just squad-mates.
    """
    reserve = FOOD_RESERVE_TURNS * piece.diet
    surplus = max(0.0, piece.food - reserve)
    if surplus <= 0:
        return

    receivers = [
        p for p in piece.pieces_in_sight(
            grid, elevations, battlefield,
            facing_only=True, allied_only=True
        )
        if p.food < p.food_capacity
    ]

    receivers.sort(key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

    for receiver in receivers:
        if surplus <= 0:
            break
        amount = min(surplus, receiver.food_capacity - receiver.food)
        receiver.food += amount
        piece.food -= amount
        surplus -= amount
```

Then refactor `Squad.feed_turn` to accept the battlefield:

```python
@patch
def feed_turn(self: Squad, grid: HexGrid, elevations: np.ndarray,
              food_tiers: np.ndarray, battlefield: list[Piece] = None):
    """One turn's food cycle. battlefield = all pieces for cross-squad giving."""
    bf = battlefield if battlefield is not None else self.pieces

    for p in self.alive:
        if p.instructions.current == Instruction.HARVEST:
            harvest(p, grid, food_tiers)

    for p in self.by_rank():
        p.give(grid, elevations, bf)

    self.pieces = [p for p in self.pieces if eat(p)]
```

And then the coordinator that ties squads together — this could live on `PieceBoard` or a new `Battlefield` concept:

```python
@patch
def all_pieces(self: PieceBoard) -> list[Piece]:
    """Flat list of every piece across all squads."""
    return [p for sq in self.squads for p in sq]

@patch
def feed_all(self: PieceBoard, food_tiers: np.ndarray):
    """Run food cycle for all squads with cross-squad visibility."""
    bf = self.all_pieces()
    for sq in self.squads:
        sq.feed_turn(self.grid, self.elevations, food_tiers, battlefield=bf)
```

The main structural change would be `PieceBoard.pieces: list[Piece]` → `PieceBoard.squads: list[Squad]`, with `all_pieces` as a property for backward compat. Want to tackle that migration, or flesh out squad assignment (who goes into which squad) first?

I love Hungarians, I was once taught by Egon Balas. I think we want two option - one facing inward as if they are feeding the queen and one facing outward so they could pretect her.

In [ ]:
#| export
class SurroundMode(Enum):
    FEED  = "feed"   # face inward — supply chain
    GUARD = "guard"  # face outward — defensive perimeter


@patch
def effective_sight(self: Piece, elevations: np.ndarray,
                    elevation_mult: float = 0.005) -> float:
    """Sight range including elevation bonus."""
    if self.location is None or self.location < 0:
        return float(self.sight)
    elev = max(0, elevations[self.location])
    return self.sight + elev * elevation_mult


def _facing_toward(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) from from_idx toward to_idx."""
    rel = grid.index_to_hexposition(to_idx, from_idx)
    # Find closest cardinal direction
    dirs = HexPosition.directions()
    best_dir = 0
    best_dot = -999
    for i, d in enumerate(dirs):
        # Dot product in cube coords
        dot = rel.q * d.q + rel.r * d.r + rel.s * d.s
        if dot > best_dot:
            best_dot = dot
            best_dir = i
    return best_dir


def _facing_away(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) pointing away from to_idx."""
    return (_facing_toward(grid, from_idx, to_idx) + 3) % 6



In [ ]:
#| export

@patch
def surround(self: Squad, target: int,
             grid: HexGrid, elevations: np.ndarray,
             mode: SurroundMode = SurroundMode.GUARD,
             max_ring: int = 3,
             countries: np.ndarray = None,
             elevation_mult: float = 0.005
             ) -> dict[str, InstructionList]:
    """Assign squad members to surround positions using Hungarian matching.
    
    Args:
        target: Hex index to surround
        grid: HexGrid
        elevations: Elevation array
        mode: FEED (face inward) or GUARD (face outward)
        max_ring: Maximum ring distance from target
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList with path + final rotation
    """
    pieces = self.alive
    if not pieces:
        return {}

    # 1. Generate candidate positions (rings 1..max_ring)
    candidates = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target)
            if idx < 0 or idx in grid.invalidRegion:
                continue
            if elevations[idx] <= 0:
                continue
            if countries is not None and countries[idx] < 0:
                continue
            candidates.append((idx, ring))

    if not candidates:
        return {}

    n_pieces = len(pieces)
    n_positions = len(candidates)

    # 2. Build cost matrix, caching TroopPaths to avoid double-pathfind
    INF = 1e9
    cost = np.full((n_pieces, n_positions), INF)
    paths: dict[tuple[int, int], TroopPath] = {}

    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        eff_sight = piece.effective_sight(elevations, elevation_mult)

        for j, (pos_idx, ring_dist) in enumerate(candidates):
            if ring_dist > eff_sight:
                continue
            path = piece.pathfind(pos_idx, grid, elevations, countries)
            if not path:
                continue
            paths[(i, j)] = path
            cost[i, j] = path.cost

    # 3. Hungarian matching
    row_ind, col_ind = linear_sum_assignment(cost)

    # 4. Generate instruction lists from cached paths
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue

        piece = pieces[i]
        path = paths[(i, j)]
        pos_idx = candidates[j][0]

        rules = path.to_rules(grid, piece.facing)

        # Desired final facing
        if mode == SurroundMode.FEED:
            desired = _facing_toward(grid, pos_idx, target)
        else:
            desired = _facing_away(grid, pos_idx, target)

        # Facing after the last move step
        end_facing = _facing_toward(grid, path[-2], path[-1])

        # Shortest rotation to desired facing
        diff = (desired - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=False)

    return assignments


Do we need TroopPaths at all for surround. does it make it cleaner?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, and it meaningfully cleans up two specific spots in `surround`:

**Currently:**
1. `pathfind` is called **twice** per piece/position pair — once to compute the cost matrix, once after matching to generate rules
2. The cost is a manual `sum(_move_cost(...))` loop even though `TroopPath.cost` already holds it

**With `TroopPath` cached during the cost build:**

```python
# 2. Build cost matrix, caching paths to avoid double-pathfind
paths: dict[tuple[int, int], TroopPath] = {}

for i, piece in enumerate(pieces):
    if piece.location is None or piece.location < 0:
        continue
    for j, (pos_idx, ring_dist) in enumerate(candidates):
        path = piece.pathfind(pos_idx, grid, elevations, countries)
        if not path:
            continue
        paths[(i, j)] = path
        cost[i, j] = path.cost          # ← already on TroopPath, no loop needed

# 4. Generate instruction lists
for i, j in zip(row_ind, col_ind):
    if cost[i, j] >= INF:
        continue
    path = paths.get((i, j))
    if path is None:
        continue

    rules = path.to_rules(grid, piece.facing)   # ← clean one-liner
    ...
```

So `TroopPath` earns its keep here because:
- **`.cost`** eliminates the manual sum
- **`.to_rules()`** cleans up the rule generation line  
- **Caching** eliminates the second `pathfind` call after matching — which was potentially expensive since Dijkstra runs again from scratch

The dict key `(i, j)` is cheap and the cache naturally expires at end of the method. Worth keeping.

can you build the refactored function?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def surround(self: Squad, target: int,
             grid: HexGrid, elevations: np.ndarray,
             mode: SurroundMode = SurroundMode.GUARD,
             max_ring: int = 3,
             countries: np.ndarray = None,
             elevation_mult: float = 0.005
             ) -> dict[str, InstructionList]:
    """Assign squad members to surround positions using Hungarian matching.
    
    Args:
        target: Hex index to surround
        grid: HexGrid
        elevations: Elevation array
        mode: FEED (face inward) or GUARD (face outward)
        max_ring: Maximum ring distance from target
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList with path + final rotation
    """
    pieces = self.alive
    if not pieces:
        return {}

    # 1. Generate candidate positions (rings 1..max_ring)
    candidates = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target)
            if idx < 0 or idx in grid.invalidRegion:
                continue
            if elevations[idx] <= 0:
                continue
            if countries is not None and countries[idx] < 0:
                continue
            candidates.append((idx, ring))

    if not candidates:
        return {}

    n_pieces = len(pieces)
    n_positions = len(candidates)

    # 2. Build cost matrix, caching TroopPaths to avoid double-pathfind
    INF = 1e9
    cost = np.full((n_pieces, n_positions), INF)
    paths: dict[tuple[int, int], TroopPath] = {}

    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        eff_sight = piece.effective_sight(elevations, elevation_mult)

        for j, (pos_idx, ring_dist) in enumerate(candidates):
            if ring_dist > eff_sight:
                continue
            path = piece.pathfind(pos_idx, grid, elevations, countries)
            if not path:
                continue
            paths[(i, j)] = path
            cost[i, j] = path.cost

    # 3. Hungarian matching
    row_ind, col_ind = linear_sum_assignment(cost)

    # 4. Generate instruction lists from cached paths
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue

        piece = pieces[i]
        path = paths[(i, j)]
        pos_idx = candidates[j][0]

        rules = path.to_rules(grid, piece.facing)

        # Desired final facing
        if mode == SurroundMode.FEED:
            desired = _facing_toward(grid, pos_idx, target)
        else:
            desired = _facing_away(grid, pos_idx, target)

        # Facing after the last move step
        end_facing = _facing_toward(grid, path[-2], path[-1])

        # Shortest rotation to desired facing
        diff = (desired - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))

        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=False)

    return assignments
```

The three wins over the previous version:

1. **Single pathfind** per piece/position pair — cached in `paths[(i, j)]` during cost-matrix build, reused after matching
2. **`path.cost`** replaces the manual `sum(_move_cost(...))` loop
3. **`path.to_rules()`** replaces the raw `InstructionList.path_to_rules(path.hexes, ...)` call

In [ ]:
# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Place the queen near center on high ground
mid = grid.middle
queen = board.add_piece(mid, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0  # hungry queen

# Scatter pawns and a knight around — various distances
offsets = [
    (-3, -2), (-4, 1), (2, -3), (3, 2), (5, 0), (-2, 4), (1, -5), (-5, -1)
]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0
    feeders.append(p)

# --- Group into a squad ---
escort = Squad(id=1, name="Queen's Escort", pieces=feeders)

print(f"Queen at hex {mid}, squad '{escort.name}' has {len(escort)} members")

# --- Surround with FEED mode ---
orders = escort.surround(queen.location, grid, elevs,
                         mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)}/{len(escort)} pieces to surround positions")
for p in escort.alive:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render ---
#board.render(show_plan=True, num_turns=8)
#show(board)


How about with a pieces overlay and the queen not sitting in water?

# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Find nearest land hex to center for queen
mid = grid.middle
queen_idx = next(
    idx
    for dist in range(10)
    for hp in ([HexPosition.origin()] if dist == 0 else HexPosition.origin().ring(dist))
    for idx in [grid.hexposition_to_index(hp, mid)]
    if idx >= 0 and idx not in grid.invalidRegion and elevs[idx] > 0
)
print(f"Queen placed at hex {queen_idx} (elev={elevs[queen_idx]:.0f})")

queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0

# Scatter feeders — skip any that land on water
offsets = [(-3,-2), (-4,1), (2,-3), (3,2), (5,0), (-2,4), (1,-5), (-5,-1)]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, queen_idx)
    if idx < 0 or idx in grid.invalidRegion or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0
    feeders.append(p)

print(f"{len(feeders)} feeders placed")

# --- Group into squad and surround ---
escort = Squad(id=1, name="Queen's Escort", pieces=feeders)
orders = escort.surround(queen.location, grid, elevs, mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)}/{len(escort)} pieces")
for p in escort.alive:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render with plan overlay ---
PieceBoardPlan(board, num_turns=8)


@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue
        
        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#333"
        g = DiagramGlyphs(color=color, size=self.grid.radius * 0.45)
        svg += g.facing_bar(c.x, c.y, last.facing)
    
    return svg


@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


In [ ]:
@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#fff"
        
        # Dark outline pass first, then bright bar on top
        g_outline = DiagramGlyphs(color="#222222",
                                  size=self.grid.radius * 0.55,
                                  stroke_width=4.5, opacity=0.9)
        g_bar = DiagramGlyphs(color="#FFFFFF",
                              size=self.grid.radius * 0.55,
                              stroke_width=2.5, opacity=1.0)
        
        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()



In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

I can't really see the bars. Are they under? should we use a different color?

so they should extend further towards the edge of the hex we also want thing rotated to face the queen

In [ ]:
@patch
def facing_bar(self: DiagramGlyphs, cx, cy, facing: int,
               length=None, offset=None, stroke=None,
               stroke_width=None, opacity=None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    import math
    length       = length       or self.size * 1.2          # wider bar
    offset       = offset       or self.size * 1.1          # pushed toward hex edge
    stroke       = stroke       or self.color
    stroke_width = stroke_width or max(2, self.size * 0.15)
    opacity      = opacity if opacity is not None else self.opacity

    # SW=0,W=1,NW=2,NE=3,E=4,SE=5 → pixel angles 120,180,240,300,0,60
    angle = math.radians(60 * facing + 120)

    bx = cx + offset * math.cos(angle)
    by = cy + offset * math.sin(angle)

    perp = angle + math.pi / 2
    half = length / 2
    x1, y1 = bx + half * math.cos(perp), by + half * math.sin(perp)
    x2, y2 = bx - half * math.cos(perp), by - half * math.sin(perp)

    return (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
            f'x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.1f}" '
            f'stroke-linecap="round" opacity="{opacity}"/>\n')


@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    r = self.grid.radius
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center

        # Size drives offset/length via the multipliers above
        g_outline = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
        g_bar     = DiagramGlyphs(color="#FFFFFF",  size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)

        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

So I think we are ready for our sightOverlay. we want to see which hexes are covered by a list of pieces. perhaps the overlay could be a dotted one similar to food, but with the kingdom color

In [ ]:
@patch
def hexes_in_sight(self: Piece, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False) -> list[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return []
    
    elev = max(0, elevations[self.location])
    effective_sight = int(self.sight + elev * elevation_mult)
    
    if facing_only:
        facing_dir = HexPosition.directions()[self.facing % 6]
        hex_positions = field_of_view(HexPosition.origin(), facing_dir, effective_sight)
        return [
            idx for hp in hex_positions
            if (idx := grid.hexposition_to_index(hp, self.location)) >= 0
        ]
    else:
        return grid.indices_in_range(self.location, effective_sight)





@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.22,
                  facing_only: bool = False,
                  allowed_rotations: int = 1) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(1.5, r * 0.10)
    spacing = max(5.0, r * 0.32)

    # Group by kingdom color
    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = piece.flag.primary if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only,
            allowed_rotations=allowed_rotations
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        # Register dot pattern as a def
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.6" '
                f'stroke-dasharray="3,2" stroke-opacity="0.4"/>\n'
            )
    return svg


@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


 think we need to work on field of view which seems to go wider that 120. in ring 1 it is fine. but in ring 2 it should go straing out from its edges but it bends

These wound up too suttble. I wonder if darkPrimary or baseComp or perhaps larger dots makes more sense.

In [ ]:
@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.35,          # up from 0.22
                  facing_only: bool = False,
                  allowed_rotations: int = 1,
                  color_attr: str = "darkPrimary"  # "primary" | "darkPrimary" | "baseComp"
                  ) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(2.5, r * 0.18)   # up from 0.10
    spacing = max(5.0, r * 0.32)

    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = getattr(piece.flag, color_attr, "#888") if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.8" '
                f'stroke-dasharray="3,2" stroke-opacity="0.5"/>\n'
            )
    return svg


In [ ]:
@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()

Lets do a demo with one piece so I can see the cone

# darkPrimary — recommended
show(NotStr(board.render_with_sight(feeders + [queen], num_turns=8)))

In [ ]:
def field_of_view(origin: HexPosition, facing: HexPosition, max_distance: int) -> list[HexPosition]:
    """Get all hexes within a 120° field of view (±60° from facing) up to max_distance.
    
    Walks the cone arc per ring — no per-hex tests needed.
    """
    left_edge = facing.rotate(1)     # start of arc (left side)
    step1 = facing.rotate(-1)        # walk from left toward center
    step2 = facing.rotate(-2)        # walk from center toward right
    
    results = []
    for r in range(1, max_distance + 1):
        current = origin + r * left_edge
        for _ in range(r):
            results.append(current)
            current = current + step1
        for _ in range(r):
            results.append(current)
            current = current + step2
        results.append(current)      # final right-edge hex
    return results


# --- Fresh board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid  = board.grid
elevs = board.elevations

# Find a nice land hex near center
mid = grid.middle
scout_idx = None
for dist in range(0, 6):
    for hp in HexPosition.origin().ring(dist):
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            scout_idx = idx
            break
    if scout_idx is not None:
        break

# Single scout, facing direction 3 (NE) so cone points up-right
scout = board.add_piece(scout_idx, PieceType.KNIGHT, facing=3, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
scout.sight = 4   # decent range

print(f"Scout at hex {scout_idx} (elev={elevs[scout_idx]:.0f}), facing=3 (NE)")
print(f"Effective sight: {scout.effective_sight(elevs):.2f}")

# Cone only — shows the 120° wedge
show(NotStr(board.render_with_sight(
    [scout],
    num_turns=1,
    facing_only=True   # ← cone mode
)))


So I am wondering about another matching algorith. we want to cover a path so that as a piece walks along it they would be fed. so we need to put pieces on the best yields "outside of the path". and then use a similar bipartite graph to match them. I think these field of view algorithms will come in handy for getting covverage.

In [ ]:
def supply_line(path: list[int], feeders: Squad,
                grid: HexGrid, elevations: np.ndarray,
                food_tiers: np.ndarray,
                min_tier: int = 2,
                max_offset: int = 2,
                countries: np.ndarray = None,
                elevation_mult: float = 0.005
                ) -> dict[str, InstructionList]:
    """Station feeders on fertile tiles along a march path.
    
    Args:
        path: Hex indices of the march route
        feeders: Squad of available pieces to assign as suppliers
        grid: HexGrid
        elevations: Elevation array
        food_tiers: Per-hex food tier array
        min_tier: Minimum food tier for a candidate station
        max_offset: Max ring distance from path for stations
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList (path + rotate + harvest/give loop)
    """
    pieces = feeders.alive
    if not pieces:
        return {}
    
    path_set = set(path)
    
    # --- Phase 1: Generate scored candidate stations ---
    # candidates[hex_idx] = (facing, tier, covered_path_hexes)
    candidates = {}
    
    for path_hex in path:
        for ring in range(1, max_offset + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, path_hex)
                if idx < 0 or idx in grid.invalidRegion:
                    continue
                if elevations[idx] <= 0:
                    continue
                if idx in path_set:
                    continue
                if int(food_tiers[idx]) < min_tier:
                    continue
                if countries is not None and countries[idx] < 0:
                    continue
                
                facing = _facing_toward(grid, idx, path_hex)
                facing_dir = HexPosition.directions()[facing]
                
                elev = max(0, elevations[idx])
                eff_sight = 3 + elev * elevation_mult
                fov = set(field_of_view(HexPosition.origin(), facing_dir, int(eff_sight)))
                
                covered = {ph for ph in path
                           if grid.index_to_hexposition(ph, idx) in fov}
                
                if not covered:
                    continue
                
                tier = int(food_tiers[idx])
                if idx not in candidates or len(covered) > len(candidates[idx][2]):
                    candidates[idx] = (facing, tier, covered)
    
    if not candidates:
        return {}
    
    # --- Phase 2: Greedy set cover → pick station positions ---
    uncovered = set(path)
    selected = []      # [(hex_idx, facing, tier), ...]
    remaining = dict(candidates)
    
    while uncovered and remaining:
        best_idx = max(remaining, key=lambda i: (
            len(remaining[i][2] & uncovered) * remaining[i][1]
        ))
        facing, tier, covered = remaining.pop(best_idx)
        newly_covered = covered & uncovered
        if not newly_covered:
            continue
        selected.append((best_idx, facing, tier))
        uncovered -= newly_covered
    
    if not selected:
        return {}
    
    # --- Phase 3: Hungarian matching (single pathfind per pair) ---
    INF = 1e9
    n_pieces = len(pieces)
    n_stations = len(selected)
    cost = np.full((n_pieces, n_stations), INF)
    paths: dict[tuple[int, int], TroopPath] = {}
    
    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        for j, (pos_idx, _, _) in enumerate(selected):
            tp = piece.pathfind(pos_idx, grid, elevations, countries)
            if not tp:
                continue
            paths[(i, j)] = tp
            cost[i, j] = tp.cost
    
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # --- Phase 4: Build instructions from cached paths ---
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        
        piece = pieces[i]
        tp = paths[(i, j)]
        _, facing, _ = selected[j]
        
        rules = tp.to_rules(grid, piece.facing)
        
        # Rotate to face the path
        end_facing = _facing_toward(grid, tp[-2], tp[-1])
        diff = (facing - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Station action loop
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return assignments


should we have a class TroopPath which is just a wrapper over list[int]. we might have multiple list[int] places?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, for a few good reasons:

1. `list[int]` is overloaded — it's also used for rules, squad ids, food tier indices
2. A path has natural derived properties you keep recomputing (`path_set`, total cost, length in turns)
3. `supply_line` + `surround` both repeatedly call `pathfind` and recompute cost — a `TroopPath` could cache it

Here's a lightweight design:

```python
@dataclass
class TroopPath:
    """A resolved hex route with cached metadata."""
    hexes: list[int]
    cost: float = 0.0          # total movement cost (sum of _move_cost steps)
    
    def __len__(self):   return len(self.hexes)
    def __iter__(self):  return iter(self.hexes)
    def __bool__(self):  return len(self.hexes) > 1   # empty or single-hex = falsy
    def __getitem__(self, i): return self.hexes[i]
    
    @cached_property
    def as_set(self) -> set[int]:
        return set(self.hexes)
    
    @property
    def start(self) -> int: return self.hexes[0]
    
    @property
    def end(self) -> int:   return self.hexes[-1]
    
    def to_rules(self, grid: HexGrid, start_facing: int = 0) -> list[int]:
        return InstructionList.path_to_rules(self.hexes, grid, start_facing=start_facing)
```

Then `pathfind` returns a `TroopPath` instead of bare `list[int]`:

```python
@patch
def pathfind(self: Piece, target: int, grid: HexGrid,
             elevations: np.ndarray,
             countries: np.ndarray = None) -> TroopPath:
    ...
    # in the heapq, track cost separately
    pq = [(0.0, start, [start], 0.0)]   # (priority, current, path, cost)
    ...
    while pq:
        cost, current, path, total_cost = heapq.heappop(pq)
        if current == target:
            return TroopPath(path, cost=total_cost)
        ...
        edge_cost = _move_cost(elevations, current, neighbor)
        heapq.heappush(pq, (cost + edge_cost, neighbor, 
                            path + [neighbor], total_cost + edge_cost))
    
    return TroopPath([])
```

Then in `supply_line` and `surround` the cost matrix fills cleanly:

```python
p = piece.pathfind(pos_idx, grid, elevations, countries)
if not p:   # falsy if empty/single-hex
    continue
cost[i, j] = p.cost   # no recomputation
```

The key payoff is that `p.cost` replaces the repeated `sum(_move_cost(...) for k in range(...))` pattern that appears in both `surround` and `supply_line`. Want to go ahead with this?

Can you refactor supply_line to take advantage of TroopPath

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
def supply_line(path: list[int], feeders: Squad,
                grid: HexGrid, elevations: np.ndarray,
                food_tiers: np.ndarray,
                min_tier: int = 2,
                max_offset: int = 2,
                countries: np.ndarray = None,
                elevation_mult: float = 0.005
                ) -> dict[str, InstructionList]:
    """Station feeders on fertile tiles along a march path.
    
    Args:
        path: Hex indices of the march route
        feeders: Squad of available pieces to assign as suppliers
        grid: HexGrid
        elevations: Elevation array
        food_tiers: Per-hex food tier array
        min_tier: Minimum food tier for a candidate station
        max_offset: Max ring distance from path for stations
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList (path + rotate + harvest/give loop)
    """
    pieces = feeders.alive
    if not pieces:
        return {}
    
    path_set = set(path)
    
    # --- Phase 1: Generate scored candidate stations ---
    # candidates[hex_idx] = (facing, tier, covered_path_hexes)
    candidates = {}
    
    for path_hex in path:
        for ring in range(1, max_offset + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, path_hex)
                if idx < 0 or idx in grid.invalidRegion:
                    continue
                if elevations[idx] <= 0:
                    continue
                if idx in path_set:
                    continue
                if int(food_tiers[idx]) < min_tier:
                    continue
                if countries is not None and countries[idx] < 0:
                    continue
                
                facing = _facing_toward(grid, idx, path_hex)
                facing_dir = HexPosition.directions()[facing]
                
                elev = max(0, elevations[idx])
                eff_sight = 3 + elev * elevation_mult
                fov = set(field_of_view(HexPosition.origin(), facing_dir, int(eff_sight)))
                
                covered = {ph for ph in path
                           if grid.index_to_hexposition(ph, idx) in fov}
                
                if not covered:
                    continue
                
                tier = int(food_tiers[idx])
                if idx not in candidates or len(covered) > len(candidates[idx][2]):
                    candidates[idx] = (facing, tier, covered)
    
    if not candidates:
        return {}
    
    # --- Phase 2: Greedy set cover → pick station positions ---
    uncovered = set(path)
    selected = []      # [(hex_idx, facing, tier), ...]
    remaining = dict(candidates)
    
    while uncovered and remaining:
        best_idx = max(remaining, key=lambda i: (
            len(remaining[i][2] & uncovered) * remaining[i][1]
        ))
        facing, tier, covered = remaining.pop(best_idx)
        newly_covered = covered & uncovered
        if not newly_covered:
            continue
        selected.append((best_idx, facing, tier))
        uncovered -= newly_covered
    
    if not selected:
        return {}
    
    # --- Phase 3: Hungarian matching (single pathfind per pair) ---
    INF = 1e9
    n_pieces = len(pieces)
    n_stations = len(selected)
    cost = np.full((n_pieces, n_stations), INF)
    paths: dict[tuple[int, int], TroopPath] = {}
    
    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        for j, (pos_idx, _, _) in enumerate(selected):
            tp = piece.pathfind(pos_idx, grid, elevations, countries)
            if not tp:
                continue
            paths[(i, j)] = tp
            cost[i, j] = tp.cost
    
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # --- Phase 4: Build instructions from cached paths ---
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        
        piece = pieces[i]
        tp = paths[(i, j)]
        _, facing, _ = selected[j]
        
        rules = tp.to_rules(grid, piece.facing)
        
        # Rotate to face the path
        end_facing = _facing_toward(grid, tp[-2], tp[-1])
        diff = (facing - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Station action loop
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return assignments
```

Three things cleaned up:

1. **Single pathfind** — `paths[(i, j)]` cache eliminates the second `piece.pathfind` call after matching
2. **`tp.cost`** replaces the manual `sum(_move_cost(...) for k in ...)` loop
3. **`tp.to_rules()`** replaces the raw `InstructionList.path_to_rules(p, ...)` call

Same pattern as `surround` now — cost matrix build + cache, then Hungarian, then generate rules from cached paths.

can you build a demo. for gameboards maybe just assume elevation determines food yield for our example

@patch
def pathfind_to_rules(self: Piece, target: int, grid: HexGrid,
                      elevations: np.ndarray,
                      countries: np.ndarray = None) -> InstructionList:
    """Find path to target and convert to movement instructions."""
    path = self.pathfind(target, grid, elevations, countries)
    if len(path) < 2:
        return InstructionList([])
    
    rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
    return InstructionList(rules, cursor=0, patrol=False)


@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> InstructionList:
    """Find nearest tile with food >= min_tier and path there + harvest."""
    start = self.location
    if start is None or start < 0:
        return InstructionList([])
    
    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    
    while pq:
        cost, current = heapq.heappop(pq)
        
        if current in visited:
            continue
        visited.add(current)
        
        if current != start and int(food_tiers[current]) >= min_tier:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            
            rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
            rules.append(Instruction.HARVEST.value)
            return InstructionList(rules, cursor=0, patrol=False)
        
        for neighbor in grid.neighborsOf(current):
            if neighbor in visited or neighbor in grid.invalidRegion:
                continue
            if elevations[neighbor] <= 0:
                continue
            if countries is not None and countries[neighbor] < 0:
                continue
            
            edge_cost = _move_cost(elevations, current, neighbor)
            if neighbor not in parent:
                parent[neighbor] = current
                heapq.heappush(pq, (cost + edge_cost, neighbor))
    
    return InstructionList([])


# --- Board setup ---
board = PieceBoard.hilly(rings=8, radius=18, seed=42)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

# Fake food tiers from elevation
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if e <= 0:      food_tiers[i] = 0
    elif e < 80:    food_tiers[i] = 6
    elif e < 140:   food_tiers[i] = 5
    elif e < 200:   food_tiers[i] = 3
    elif e < 280:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# --- Find start and target on the NORTH side (above river) ---
def find_land(offsets):
    for hp in offsets:
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            return idx
    return None

start_idx  = find_land([HexPosition(-5, -3, 8), HexPosition(-4, -3, 7), HexPosition(-3, -2, 5)])
target_idx = find_land([HexPosition(5, -3, -2), HexPosition(4, -3, -1), HexPosition(3, -2, -1)])

print(f"Start: hex {start_idx} (elev={elevs[start_idx]:.0f})")
print(f"Target: hex {target_idx} (elev={elevs[target_idx]:.0f})")

# --- Queen ---
queen = board.add_piece(start_idx, PieceType.QUEEN, facing=4, country_id=1,
                        instructions=InstructionList([]))
queen.food = 3.0
march_path = queen.pathfind(target_idx, grid, elevs)
queen.instructions = queen.pathfind_to_rules(target_idx, grid, elevs)
print(f"March path: {len(march_path)} hexes, {len(queen.instructions.rules)} instructions")

# --- Scatter feeders on the NORTH side ---
feeder_offsets = [
    (-4, -1), (-3, -3), (-1, -2), (0, -3), (1, -1),
    (2, -3), (3, -1), (-2, -1), (4, -2), (-1, -4)
]
feeders = []
for dq, dr in feeder_offsets:
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    p = board.add_piece(idx, PieceType.PAWN, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 5.0
    p.sight = 3
    feeders.append(p)

print(f"{len(feeders)} feeders available")

# --- Wrap in Squad and run supply line ---
feeder_squad = Squad(id=1, name="Supply Corps", pieces=feeders)

orders = supply_line(march_path.hexes, feeder_squad, grid, elevs, food_tiers,
                     min_tier=3, max_offset=2)

print(f"\nAssigned {len(orders)} feeders:")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name}: {len(p.instructions.rules)} instrs")

# --- Render ---
assigned = [p for p in feeders if p.id in orders]
show(NotStr(board.render_with_sight(
    assigned + [queen],
    num_turns=12,
    facing_only=True
)))




##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Use the `TroopPath`-based versions — they're thinner and more composable. Here's the reasoning:

**`pathfind_to_rules`** — the TroopPath version is a clean 3-liner:

```python
@patch
def pathfind_to_rules(self: Piece, target: int, grid: HexGrid,
                      elevations: np.ndarray,
                      countries: np.ndarray = None) -> InstructionList:
    """Thin wrapper: path to target → InstructionList."""
    path = self.pathfind(target, grid, elevations, countries)
    if not path:
        return InstructionList()
    return InstructionList(path.to_rules(grid, self.facing), cursor=0, patrol=False)
```

The other version manually calls `InstructionList.path_to_rules(path, grid, ...)` which is exactly what `TroopPath.to_rules` already wraps, and checks `len(path) < 2` which is what `TroopPath.__bool__` does.

**`pathfind_to_food`** — keep the one that returns `TroopPath`, not `InstructionList`. The later version bakes in the `HARVEST` append and returns `InstructionList`, which kills reuse. Better pattern:

```python
@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> TroopPath:
    """Nearest reachable tile with food >= min_tier."""
    # ... Dijkstra returning TroopPath ...
```

Then the caller decides what to append:

```python
tp = piece.pathfind_to_food(grid, elevs, food_tiers)
if tp:
    rules = tp.to_rules(grid, piece.facing)
    rules.append(Instruction.HARVEST.value)        # caller's choice
    piece.instructions = InstructionList(rules, patrol=True)
```

**What to delete:** the later cell that re-defines both functions with inline Dijkstra / `InstructionList` returns. Those duplicate logic that now lives in `TroopPath.to_rules()` and `Piece.pathfind()`.

The rule of thumb: `pathfind*` methods return `TroopPath`. Converting to `InstructionList` happens at the call site or in thin wrappers like `pathfind_to_rules`. This keeps `supply_line` and `surround` working with the same `tp.cost` / `tp.to_rules()` pattern.

can you fix the last demo

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Two issues: `feeders` is a bare `list` not a `Squad`, and `march_path` is a `TroopPath` — pass `.hexes` for the type hint:

```python
# --- Board setup ---
board = PieceBoard.hilly(rings=8, radius=18, seed=42)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

# Fake food tiers from elevation
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if e <= 0:      food_tiers[i] = 0
    elif e < 80:    food_tiers[i] = 6
    elif e < 140:   food_tiers[i] = 5
    elif e < 200:   food_tiers[i] = 3
    elif e < 280:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# --- Find start and target on the NORTH side (above river) ---
def find_land(offsets):
    for hp in offsets:
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            return idx
    return None

start_idx  = find_land([HexPosition(-5, -3, 8), HexPosition(-4, -3, 7), HexPosition(-3, -2, 5)])
target_idx = find_land([HexPosition(5, -3, -2), HexPosition(4, -3, -1), HexPosition(3, -2, -1)])

print(f"Start: hex {start_idx} (elev={elevs[start_idx]:.0f})")
print(f"Target: hex {target_idx} (elev={elevs[target_idx]:.0f})")

# --- Queen ---
queen = board.add_piece(start_idx, PieceType.QUEEN, facing=4, country_id=1,
                        instructions=InstructionList([]))
queen.food = 3.0
march_path = queen.pathfind(target_idx, grid, elevs)
queen.instructions = queen.pathfind_to_rules(target_idx, grid, elevs)
print(f"March path: {len(march_path)} hexes, {len(queen.instructions.rules)} instructions")

# --- Scatter feeders on the NORTH side ---
feeder_offsets = [
    (-4, -1), (-3, -3), (-1, -2), (0, -3), (1, -1),
    (2, -3), (3, -1), (-2, -1), (4, -2), (-1, -4)
]
feeders = []
for dq, dr in feeder_offsets:
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    p = board.add_piece(idx, PieceType.PAWN, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 5.0
    p.sight = 3
    feeders.append(p)

print(f"{len(feeders)} feeders available")

# --- Wrap in Squad and run supply line ---
feeder_squad = Squad(id=1, name="Supply Corps", pieces=feeders)

orders = supply_line(march_path.hexes, feeder_squad, grid, elevs, food_tiers,
                     min_tier=3, max_offset=2)

print(f"\nAssigned {len(orders)} feeders:")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name}: {len(p.instructions.rules)} instrs")

# --- Render ---
assigned = [p for p in feeders if p.id in orders]
show(NotStr(board.render_with_sight(
    assigned + [queen],
    num_turns=12,
    facing_only=True
)))
```

Two fixes:
- `feeders` → `Squad(id=1, name="Supply Corps", pieces=feeders)` so `.alive` works
- `march_path` → `march_path.hexes` since `supply_line` expects `list[int]`